In [11]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from NEAT1 import Genotipo, Poblacion

class ClasificacionGenotipo(Genotipo):

    def __init__(self, num_entradas=None, num_salidas=1):
        super().__init__()
        self.num_entradas = num_entradas
        self.num_salidas = num_salidas
        self.fitness = 0.0
    
    def inicializar(self, num_entradas, num_salidas=1):
        self.num_entradas = num_entradas
        self.num_salidas = num_salidas
        return self
    
    def predecir(self, entrada):
        salida = self.fenotipo(entrada)
        return 1 if salida[0] >= 0.5 else 0
    
    def calcular_fitness(self, x, y):
        aciertos = 0
        for i in range(len(x)):
            prediccion = self.predecir(x[i])
            if prediccion == y[i]:
                aciertos += 1
        self.fitness = aciertos / len(x)
        return self.fitness


def preprocesar_datos():
    data = load_breast_cancer()
    x = data.data
    y = data.target
    y = 1 - y  #maligno->1, benigno->0
    scaler = StandardScaler()  #para tipificar los datos
    x_tipificada = scaler.fit_transform(x)  #calcula la media y desviacion tipica de x y la tipifica
    print(f"Los datos recogen {len(x)} observaciones con {x.shape[1]} caracteristicas cada una")
    print(f"Clases: {sum(y==0)} benignos, {sum(y==1)} malignos")
    return x_tipificada, y


def evaluar_modelo(x, y, x_train, x_test, y_train, y_test, num_generaciones=25):
    poblacion = Poblacion(N=150,genotipos=lambda: ClasificacionGenotipo(num_entradas=x_train.shape[1], #numero de caracteristicas por observacion
            num_salidas=1),num_entrada=x_train.shape[1],num_salida=1)
    mejor_fitnesshis = []
    
    for generacion in range(num_generaciones):
        for i in poblacion.individuos:
            i.calcular_fitness(x_train, y_train)
        poblacion.especiacion(delta_t=3.0)
        mejor_fit = max(ind.fitness for ind in poblacion.individuos)
        mejor_fitnesshis.append(mejor_fit)
        if generacion < num_generaciones - 1:  #reproduccion excepto en la ultima generacion
            poblacion.reproduccion()
        if generacion % 10 == 0:  #cada 10 generaciones devuelve el mejor fitness para poder seguir la evolucion
            print(f"En la generacion {generacion}, el mejor fitness es {mejor_fit:.4f}")
    mejor_individuo = max(poblacion.individuos, key=lambda x: x.fitness) #el mejor fitness de la ultima generacion

    aciertos= 0
    y_pred = []
    for i in range(len(x_test)):
        pred = mejor_individuo.predecir(x_test[i]) #predicciones del mejor individuo para la muestra test
        y_pred.append(pred)
        if pred == y_test[i]:
            aciertos+= 1 #se considera u  acierto si la prediccion coincide con el valor real 
    
    #Matriz de confusion
    VP = sum(1 for i in range(len(y_test)) if y_pred[i] == 1 and y_test[i] == 1)
    VN = sum(1 for i in range(len(y_test)) if y_pred[i] == 0 and y_test[i] == 0)
    FP = sum(1 for i in range(len(y_test)) if y_pred[i] == 1 and y_test[i] == 0)
    FN = sum(1 for i in range(len(y_test)) if y_pred[i] == 0 and y_test[i] == 1)
    #medidas de rendimiento
    exactitud= (VP + VN) / len(x_test)
    precision = VP / (VP + FP) if (VP + FP) > 0 else 0
    sensibilidad = VP / (VP + FN) if (VP + FN) > 0 else 0
    f1 = 2 * (precision * sensibilidad) / (precision + sensibilidad) if (precision + sensibilidad) > 0 else 0
    
    medidas = {'exactitud': exactitud,'precision': precision,'sensibilidad': sensibilidad,'f1_score': f1,'VP': VP,'VN': VN,'FP': FP,'FN': FN,'mejor_fitnesshis': mejor_fitnesshis,'num_nodos': len(mejor_individuo.nodos),'num_conexiones': len(mejor_individuo.conexiones)}
    return mejor_individuo, medidas


def validacion_cruzada(x, y, n_pliegues=10, num_generaciones=25):
    val = StratifiedKFold(n_splits=n_pliegues, shuffle=True, random_state=42)
    resultados = []
    mejor_inglobal = None
    mejor_fitglobal = -1
    
    print(f"\nIniciando validacion cruzada con {n_pliegues} pliegues...")
    print(" ")
    
    for pliegue, (train_id, test_id) in enumerate(val.split(x, y)):
        print(f"\n  Pliegue {pliegue + 1} de {n_pliegues}") 
        x_train, x_test = x[train_id], x[test_id]
        y_train, y_test = y[train_id], y[test_id]
        mejor_individuo, medidas = evaluar_modelo(x, y, x_train, x_test, y_train, y_test,num_generaciones=num_generaciones)
        resultados.append(medidas)
        
        if medidas['exactitud'] > mejor_fitglobal:
            mejor_fitglobal = medidas['exactitud']
            mejor_inglobal = mejor_individuo
        
        print(f"  Exactitud: {medidas['exactitud']:.4f}")
        print(f"  Precision: {medidas['precision']:.4f}")
        print(f"  Sensibilidad: {medidas['sensibilidad']:.4f}")
        print(f"  F1-Score: {medidas['f1_score']:.4f}")
        print(f"  Nodos: {medidas['num_nodos']}, Conexiones: {medidas['num_conexiones']}")
    return resultados, mejor_inglobal


def imprimir_resultados(resultados):
    print(" ")
    print("Resultados de la validacion cruzada")
    print(" ")
    
    medidas_media = {
        'exactitud': np.mean([r['exactitud'] for r in resultados]),
        'precision': np.mean([r['precision'] for r in resultados]),
        'sensibilidad': np.mean([r['sensibilidad'] for r in resultados]),
        'f1_score': np.mean([r['f1_score'] for r in resultados]),
        'nodos': np.mean([r['num_nodos'] for r in resultados]),
        'conexiones': np.mean([r['num_conexiones'] for r in resultados])}
    
    medidas_desv = {
        'exactitud': np.std([r['exactitud'] for r in resultados]),
        'precision': np.std([r['precision'] for r in resultados]),
        'sensibilidad': np.std([r['sensibilidad'] for r in resultados]),
        'f1_score': np.std([r['f1_score'] for r in resultados]),
        'nodos': np.std([r['num_nodos'] for r in resultados]),
        'conexiones': np.std([r['num_conexiones'] for r in resultados])}
    
    print("\n Media medidas (desviación típica):")
    print(f"  Exactitud:  {medidas_media['exactitud']:.4f} ({medidas_desv['exactitud']:.4f})")
    print(f"  Precision: {medidas_media['precision']:.4f} ({medidas_desv['precision']:.4f})")
    print(f"  Sensibilidad: {medidas_media['sensibilidad']:.4f} ({medidas_desv['sensibilidad']:.4f})")
    print(f"  F1-Score:  {medidas_media['f1_score']:.4f} ({medidas_desv['f1_score']:.4f})")


def main():
    print(" ")
    print("Diagnostico de tumores malignos mediante el metodo NEAT")
    print("Dataset: Wisconsin Diagnostic Breast Cancer (WDBC)")
    print(" ")
    
    x, y = preprocesar_datos()
    
    resultados, mejor_individuo = validacion_cruzada(x, y,n_pliegues=10,num_generaciones=25) 
    imprimir_resultados(resultados)
    
    print(" ")
    print("Mejor individuo")
    print(" ")
    print(f"  Fitness: {mejor_individuo.fitness:.4f}")
    print(f"  Nodos totales: {len(mejor_individuo.nodos)}")
    print(f"  Conexiones totales: {len(mejor_individuo.conexiones)}")


if __name__ == "__main__":
    main()

#    Diagnostico de tumores malignos mediante el metodo NEAT
#Dataset: Wisconsin Diagnostic Breast Cancer (WDBC)
 
#Los datos recogen 569 observaciones con 30 caracteristicas cada una
#Clases: 357 benignos, 212 malignos

#Iniciando validacion cruzada con 10 pliegues...
 

#  Pliegue 1 de 10
#En la generacion 0, el mejor fitness es 0.8965
#En la generacion 10, el mejor fitness es 0.9805
#En la generacion 20, el mejor fitness es 0.9824
#  Exactitud: 0.9825
#  Precision: 0.9565
#  Sensibilidad: 1.0000
#  F1-Score: 0.9778
#  Nodos: 31, Conexiones: 30

#  Pliegue 2 de 10
#En la generacion 0, el mejor fitness es 0.9082
#En la generacion 10, el mejor fitness es 0.9844
#En la generacion 20, el mejor fitness es 0.9883
#  Exactitud: 0.9825
#  Precision: 0.9565
#  Sensibilidad: 1.0000
#  F1-Score: 0.9778
#  Nodos: 31, Conexiones: 30

#  Pliegue 3 de 10
#En la generacion 0, el mejor fitness es 0.9160
#En la generacion 10, el mejor fitness es 0.9805
#En la generacion 20, el mejor fitness es 0.9805
#  Exactitud: 0.9649
#  Precision: 0.9130
#  Sensibilidad: 1.0000
#  F1-Score: 0.9545
#  Nodos: 31, Conexiones: 30

#  Pliegue 4 de 10
#En la generacion 0, el mejor fitness es 0.9355
#En la generacion 10, el mejor fitness es 0.9785
#En la generacion 20, el mejor fitness es 0.9863
#  Exactitud: 0.9123
#  Precision: 0.9444
#  Sensibilidad: 0.8095
#  F1-Score: 0.8718
#  Nodos: 31, Conexiones: 30

#  Pliegue 5 de 10
#En la generacion 0, el mejor fitness es 0.9023
#En la generacion 10, el mejor fitness es 0.9863
#En la generacion 20, el mejor fitness es 0.9863
#  Exactitud: 0.9649
#  Precision: 0.9524
#  Sensibilidad: 0.9524
#  F1-Score: 0.9524
#  Nodos: 31, Conexiones: 30

#  Pliegue 6 de 10
#En la generacion 0, el mejor fitness es 0.9199
#En la generacion 10, el mejor fitness es 0.9824
#En la generacion 20, el mejor fitness es 0.9883
#  Exactitud: 0.9474
#  Precision: 1.0000
#  Sensibilidad: 0.8571
#  F1-Score: 0.9231
#  Nodos: 33, Conexiones: 34

#  Pliegue 7 de 10
#En la generacion 0, el mejor fitness es 0.9062
#En la generacion 10, el mejor fitness es 0.9707
#En la generacion 20, el mejor fitness es 0.9824
#  Exactitud: 1.0000
#  Precision: 1.0000
#  Sensibilidad: 1.0000
#  F1-Score: 1.0000
#  Nodos: 31, Conexiones: 30

#  Pliegue 8 de 10
#En la generacion 0, el mejor fitness es 0.9238
#En la generacion 10, el mejor fitness es 0.9824
#En la generacion 20, el mejor fitness es 0.9844
#  Exactitud: 0.9649
#  Precision: 0.9130
#  Sensibilidad: 1.0000
#  F1-Score: 0.9545
#  Nodos: 31, Conexiones: 30

#  Pliegue 9 de 10
#En la generacion 0, el mejor fitness es 0.8906
#En la generacion 10, el mejor fitness es 0.9766
#En la generacion 20, el mejor fitness es 0.9805
#  Exactitud: 0.9649
#  Precision: 0.9524
#  Sensibilidad: 0.9524
#  F1-Score: 0.9524
#  Nodos: 31, Conexiones: 30

#  Pliegue 10 de 10
#En la generacion 0, el mejor fitness es 0.9240
#En la generacion 10, el mejor fitness es 0.9766
#En la generacion 20, el mejor fitness es 0.9825
#  Exactitud: 1.0000
#  Precision: 1.0000
#  Sensibilidad: 1.0000
#  F1-Score: 1.0000
#  Nodos: 32, Conexiones: 32
 
#Resultados de la validacion cruzada
 

# Media medidas (desviación típica):
#  Exactitud:  0.9684 (0.0246)
#  Precision: 0.9588 (0.0309)
#  Sensibilidad: 0.9571 (0.0655)
#  F1-Score:  0.9564 (0.0361)
 
#Mejor individuo
 
#  Fitness: 0.9844
#  Nodos totales: 31
#  Conexiones totales: 30

 
Diagnostico de tumores malignos mediante el metodo NEAT
Dataset: Wisconsin Diagnostic Breast Cancer (WDBC)
 
Los datos recogen 569 observaciones con 30 caracteristicas cada una
Clases: 357 benignos, 212 malignos

Iniciando validacion cruzada con 10 pliegues...
 

  Pliegue 1 de 10
En la generacion 0, el mejor fitness es 0.8965
En la generacion 10, el mejor fitness es 0.9805
En la generacion 20, el mejor fitness es 0.9824
  Exactitud: 0.9825
  Precision: 0.9565
  Sensibilidad: 1.0000
  F1-Score: 0.9778
  Nodos: 31, Conexiones: 30

  Pliegue 2 de 10
En la generacion 0, el mejor fitness es 0.9082
En la generacion 10, el mejor fitness es 0.9844
En la generacion 20, el mejor fitness es 0.9883
  Exactitud: 0.9825
  Precision: 0.9565
  Sensibilidad: 1.0000
  F1-Score: 0.9778
  Nodos: 31, Conexiones: 30

  Pliegue 3 de 10
En la generacion 0, el mejor fitness es 0.9160
En la generacion 10, el mejor fitness es 0.9805
En la generacion 20, el mejor fitness es 0.9805
  Exactitud: 0.9649
  Precisi